In [1]:
from Trips_Extractor import extract_trips_from_pdf
from Lines_Extractor import parse_line_report_pdf
from pprint import pprint

In [2]:
lines_pdf = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Lines edge case.pdf"
trips_pdf = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Trips.pdf"

lines_pdf = "A:/Github Repos/Personal Projects/UPS-project/SamplePDFs/2507 Lines.pdf"
trips_pdf = "A:/Github Repos/Personal Projects/UPS-project/SamplePDFs/2507 Trips.pdf"

lines = parse_line_report_pdf(lines_pdf)
trips = extract_trips_from_pdf(trips_pdf)

bid_period_info = {x: lines[x] for x in ('bid_period_date_range','pay_period_date_ranges')}

In [3]:

pprint(bid_period_info)


{'bid_period_date_range': {'end': '2025-11-30', 'start': '2025-11-02'},
 'pay_period_date_ranges': {'PP1': {'end': '2025-11-29', 'start': '2025-11-02'},
                            'PP2': {'end': '2025-12-27',
                                    'start': '2025-11-30'}}}


In [4]:
bid_period_start = lines['bid_period_date_range'].get('start')
bid_period_end = lines['bid_period_date_range'].get('end')

In [5]:
from pprint import pprint
pprint(lines)

{'bid_period_date_range': {'end': '2025-11-30', 'start': '2025-11-02'},
 'lines': [{'line_number': 1,
            'pay_periods': [{'BT': '44:01',
                             'CT': '72:00',
                             'DD': '12',
                             'DO': '12',
                             'assignments': [{'date': '2025-11-04',
                                              'start_time': '07:15',
                                              'type': 'trip',
                                              'value': 1},
                                             {'date': '2025-11-05',
                                              'start_time': '07:15',
                                              'type': 'trip',
                                              'value': 1},
                                             {'date': '2025-11-06',
                                              'start_time': '07:15',
                                              'type': 'trip',
             

In [6]:
from datetime import datetime, timedelta
import re


TRIP_TYPES = {"trip", "trips"}


def parse_duration(value):
    """
    Converts:
        '20h55' -> timedelta(hours=20, minutes=55)
        '66h51' -> timedelta(hours=66, minutes=51)
        '-'     -> None
    """
    if value is None or value == "-":
        return None

    match = re.match(r"^(\d+)h(\d{2})$", str(value).strip())

    if not match:
        return None

    hours = int(match.group(1))
    minutes = int(match.group(2))

    return timedelta(hours=hours, minutes=minutes)


def datetime_at_or_after(reference_dt, time_value):
    if reference_dt is None:
        raise ValueError("reference_dt is None")

    if time_value is None:
        raise ValueError("time_value is None")

    hour, minute = map(int, time_value.split(":"))

    candidate = reference_dt.replace(
        hour=hour,
        minute=minute,
        second=0,
        microsecond=0
    )

    while candidate < reference_dt:
        candidate += timedelta(days=1)

    return candidate

def unique_preserve_order(items):
    result = []

    for item in items:
        if item not in result:
            result.append(item)

    return result


def extract_route_flags(flight):
    flags = []

    if flight.get("route_flags"):
        flags.extend(flight["route_flags"])

    if flight.get("route_flag"):
        flags.append(flight["route_flag"])

    flight_text = str(flight.get("flight", "")).upper().strip()
    route_raw = str(flight.get("route_raw", "")).upper().strip()

    # Deadhead detection
    # Examples:
    # 'DH AA194'    -> 'DH AA'
    # 'DH UPS396'   -> 'DH UPS'
    # 'DH LH1844'   -> 'DH LH'
    # 'DH WN1782-2' -> 'DH WN'
    dh_match = re.match(r"^DH\s+([A-Z]+)", flight_text)

    if dh_match:
        carrier = dh_match.group(1)
        flags.append(f"DH {carrier}")

        # Bus detection
        if "BUS" in flight_text or "BUS" in route_raw:
            flags.append("BUS")

        return unique_preserve_order(flags)


def build_master_assignment(assignment, trips):
    assignment_type = assignment.get("type")

    # -------------------------
    # Non-trip assignment
    # -------------------------
    if assignment_type not in TRIP_TYPES:
        code = assignment.get("value")

        if code is None:
            code = assignment_type

        return {
            "date": assignment.get("date"),
            "code": code
        }

    # -------------------------
    # Trip assignment
    # -------------------------
    trip_id = assignment.get("value")
    trip = trips.get(trip_id)

    if trip is None:
        return {
            "trip_id": trip_id,
            "date": assignment.get("date"),
            "error": f"Trip {trip_id} not found in trips dictionary",
            "flights": []
        }

    assignment_date = assignment.get("date")
    assignment_start_time = assignment.get("start_time")

    if assignment_date is None or assignment_start_time is None:
        return {
            "trip_id": trip_id,
            "premium": trip.get("premium"),
            "tafb": trip.get("tafb"),
            "total_days_gone": None,
            "error": "Missing assignment date or start_time",
            "flights": []
        }

    current_dt = datetime.strptime(
        f"{assignment_date} {assignment_start_time}",
        "%Y-%m-%d %H:%M"
    )

    flight_records = []

    trip_start_dt = None
    trip_end_dt = None

    blocks = trip.get("blocks", [])

    for block_index, block in enumerate(blocks, start=1):
        block_start_time = block.get("start")

        if current_dt is None:
            raise ValueError(
                f"current_dt is None before block {block_index} "
                f"of trip {trip_id}. Previous block probably had missing end time."
            )

        if block_index == 1:
            block_start_dt = current_dt
        else:
            block_start_dt = datetime_at_or_after(current_dt, block_start_time)

        last_flight_end_dt = block_start_dt
        flights = block.get("flights", [])

        for flight_index, flight in enumerate(flights):
            is_first_flight_in_block = flight_index == 0
            is_last_flight_in_block = flight_index == len(flights) - 1

            if is_first_flight_in_block:
                flight_start_dt = datetime_at_or_after(
                    block_start_dt,
                    flight.get("start")
                )
            else:
                flight_start_dt = datetime_at_or_after(
                    last_flight_end_dt,
                    flight.get("start")
                )

            flight_end_dt = datetime_at_or_after(
                flight_start_dt,
                flight.get("end")
            )

            if trip_start_dt is None:
                trip_start_dt = flight_start_dt

            trip_end_dt = flight_end_dt

            record = {
                "start_date": flight_start_dt.date().isoformat(),
                "departure": flight.get("departure"),

                "end_date": flight_end_dt.date().isoformat(),
                "arrival": flight.get("arrival"),

                "route_flags": extract_route_flags(flight),

                "rest": block.get("rest") if is_last_flight_in_block and block.get("rest") != "-" else None
            }

            flight_records.append(record)
            last_flight_end_dt = flight_end_dt

        block_end_time = block.get("end")

        if block_end_time is None:
            raise ValueError(
                f"Missing block end time in trip {trip_id}, block {block_index}"
            )

        block_end_dt = datetime_at_or_after(
            last_flight_end_dt,
            block_end_time
        )

        rest_td = parse_duration(block.get("rest"))

        if rest_td is not None:
            current_dt = block_end_dt + rest_td
        else:
            current_dt = block_end_dt

    if trip_start_dt is not None and trip_end_dt is not None:
        total_days_gone = (
            trip_end_dt.date() - trip_start_dt.date()
        ).days + 1
    else:
        total_days_gone = None

    return {
        "trip_id": trip_id,
        "premium": trip.get("premium"),
        "tafb": trip.get("tafb"),
        "total_days_gone": total_days_gone,
        "flights": flight_records
    }

In [7]:
def minutes_to_decimal_hours(total_minutes, decimals=2):
    """
    Converts:
        4350 -> 72.5
        2559 -> 42.65
    """
    return round(total_minutes / 60, decimals)

def hhmm_to_minutes(value):
    """
    Converts:
        '42:39' -> 2559 minutes
        None    -> 0
    """
    if value is None:
        return 0

    hours, minutes = value.split(":")
    return int(hours) * 60 + int(minutes)


def minutes_to_hhmm(total_minutes):
    """
    Converts:
        2559 -> '42:39'
    """
    hours = total_minutes // 60
    minutes = total_minutes % 60
    return f"{hours}:{minutes:02d}"

def creating_master_line(trips, lines):
    master_lines = {}

    for line in lines["lines"]:

        total_BT_minutes = 0
        total_CT_minutes = 0
        total_DD = 0
        total_DO = 0
        total_premium = 0.0

        line_num = line["line_number"]

        master_lines[line_num] = {
            "tot_BT": None,
            "tot_CT": None,
            "tot_DD": None,
            "tot_DO": None,
            "tot_Premium": None,
            "PPs": []
        }

        for pp in line["pay_periods"]:
            pp_DD = int(pp.get("DD") or 0)

            if pp.get("DO") is None:
                pp_DO = 28 - pp_DD
            else:
                pp_DO = int(pp.get("DO"))

            total_BT_minutes += hhmm_to_minutes(pp.get("BT"))
            total_CT_minutes += hhmm_to_minutes(pp.get("CT"))
            total_DD += pp_DD
            total_DO += pp_DO

            master_pp = {
                "pp": pp.get("pp"),
                "BT": pp.get("BT"),
                "CT": pp.get("CT"),
                "DD": pp_DD,
                "DO": pp_DO,
                "assignments": []
            }

            for assignment in pp["assignments"]:
                master_assignment = build_master_assignment(assignment, trips)
                master_pp["assignments"].append(master_assignment)

                premium = master_assignment.get("premium")

                if premium is not None:
                    total_premium += float(premium)

            master_lines[line_num]["PPs"].append(master_pp)

        master_lines[line_num]["tot_BT"] = minutes_to_decimal_hours(total_BT_minutes)
        master_lines[line_num]["tot_CT"] = minutes_to_decimal_hours(total_CT_minutes)
        master_lines[line_num]["tot_DD"] = total_DD
        master_lines[line_num]["tot_DO"] = total_DO
        master_lines[line_num]["tot_Premium"] = total_premium

    return master_lines

master_lines = creating_master_line(trips, lines)
pprint(master_lines)

{1: {'PPs': [{'BT': '44:01',
              'CT': '72:00',
              'DD': 12,
              'DO': 12,
              'assignments': [{'flights': [{'arrival': 'EWR',
                                            'departure': 'SDF',
                                            'end_date': '2025-11-04',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2025-11-04'},
                                           {'arrival': 'SDF',
                                            'departure': 'EWR',
                                            'end_date': '2025-11-04',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2025-11-04'}],
                               'premium': 0.0,
                               'tafb': '8h51',
               

In [18]:
# Original blockiness

from datetime import date, timedelta


def add_blockiness_scores(master_lines, bid_period_info):
    """
    Adds one top-level key to each line:

        line["blockiness_score"]

    Uses bid_period_info instead of bid_start / bid_end.

    bid_period_info format:

        {
            "bid_period_date_range": {
                "start": "2023-05-21",
                "end": "2023-07-16"
            },
            "pay_period_date_ranges": {
                "PP1": {
                    "start": "2023-05-21",
                    "end": "2023-06-17"
                },
                "PP2": {
                    "start": "2023-06-18",
                    "end": "2023-07-15"
                }
            }
        }

    Scoring:
        - Each PP gets its own score.
        - Final line score = average of PP scores.

    Preference order:
        TRIP > VTO > RB > RA > SB > SA > VOR

    Method:
        PP score = category_multiplier * blockiness_bonus
        except VTO, which currently uses a fixed score.
    """

    pay_period_ranges = bid_period_info["pay_period_date_ranges"]

    category_base_scores = {
        "TRIP": 700,
        "VTO": 600,
        "RB": 500,
        "RA": 400,
        "SB": 300,
        "SA": 200,
        "VOR": 100,
        "UNKNOWN": 0,
    }

    code_preference_order = ["VTO", "RB", "RA", "SB", "SA", "VOR"]
    measurable_codes = {"RB", "RA", "SB", "SA", "VOR"}

    for line_number, line in master_lines.items():

        pp_scores = []

        for pp_index, pp in enumerate(line["PPs"]):

            # --------------------------------------------------------
            # 1. Get PP date range from bid_period_info
            # --------------------------------------------------------
            pp_name = pp.get("pp", f"PP{pp_index + 1}")

            if pp_name not in pay_period_ranges:
                pp_scores.append(0)
                continue

            pp_start = date.fromisoformat(pay_period_ranges[pp_name]["start"])
            pp_end = date.fromisoformat(pay_period_ranges[pp_name]["end"])

            trip_blocks = []
            code_dates = {}

            # --------------------------------------------------------
            # 2. Read assignments using your actual master_lines format
            # --------------------------------------------------------
            for assignment in pp["assignments"]:

                # Normal trip assignment
                if "flights" in assignment:

                    start_dates = []
                    end_dates = []

                    for flight in assignment["flights"]:
                        start_dates.append(date.fromisoformat(flight["start_date"]))
                        end_dates.append(date.fromisoformat(flight["end_date"]))

                    trip_start = min(start_dates)
                    trip_end = max(end_dates)

                    trip_blocks.append({
                        "start_date": trip_start,
                        "end_date": trip_end,
                        "days_gone": assignment["total_days_gone"],
                    })

                # Code assignment: {'code': 'VTO', 'date': '2023-05-21'}
                elif "code" in assignment:

                    code = assignment["code"]
                    code_date = date.fromisoformat(assignment["date"])

                    code_dates.setdefault(code, []).append(code_date)

            # --------------------------------------------------------
            # 3. Determine PP category and work blocks
            # --------------------------------------------------------
            if trip_blocks:
                pp_category = "TRIP"
                work_blocks = trip_blocks

            else:
                pp_category = "UNKNOWN"

                for code in code_preference_order:
                    if code in code_dates:
                        pp_category = code
                        break

                work_blocks = []

                # VTO is time off, so it has no work blocks.
                if pp_category == "VTO":
                    pass

                # RB / RA / SB / SA / VOR can be measured as blocks.
                elif pp_category in measurable_codes:

                    all_code_work_dates = []

                    for code, dates in code_dates.items():
                        if code in measurable_codes:
                            all_code_work_dates.extend(dates)

                    all_code_work_dates = sorted(set(all_code_work_dates))

                    # Group consecutive code dates into blocks.
                    if all_code_work_dates:
                        block_start = all_code_work_dates[0]
                        previous_date = all_code_work_dates[0]

                        for current_date in all_code_work_dates[1:]:

                            if current_date == previous_date + timedelta(days=1):
                                previous_date = current_date
                            else:
                                days_gone = (previous_date - block_start).days + 1

                                work_blocks.append({
                                    "start_date": block_start,
                                    "end_date": previous_date,
                                    "days_gone": days_gone,
                                })

                                block_start = current_date
                                previous_date = current_date

                        days_gone = (previous_date - block_start).days + 1

                        work_blocks.append({
                            "start_date": block_start,
                            "end_date": previous_date,
                            "days_gone": days_gone,
                        })

            base_score = category_base_scores.get(pp_category, 0)

            # --------------------------------------------------------
            # 4. Special handling for VTO
            # --------------------------------------------------------
            if pp_category == "VTO":
                # VTO usually covers the whole PP.
                # Treat it as one big clean off block.
                vto_fixed_score = 60

                pp_scores.append(base_score + vto_fixed_score) # Change this line to multiplication if desired
                continue

            # --------------------------------------------------------
            # 5. Calculate blockiness bonus
            # --------------------------------------------------------
            work_blocks.sort(key=lambda block: block["start_date"])

            if not work_blocks:
                pp_scores.append(base_score)
                continue

            days_gone_list = [
                block["days_gone"]
                for block in work_blocks
            ]

            days_between_trips_list = []

            # PP start to first work block
            first_gap = (work_blocks[0]["start_date"] - pp_start).days
            days_between_trips_list.append(max(first_gap, 0))

            # Between work blocks
            for i in range(1, len(work_blocks)):
                previous_end = work_blocks[i - 1]["end_date"]
                next_start = work_blocks[i]["start_date"]

                gap = (next_start - previous_end).days
                days_between_trips_list.append(max(gap, 0))

            # Last work block to PP end
            last_gap = (pp_end - work_blocks[-1]["end_date"]).days
            days_between_trips_list.append(max(last_gap, 0))

            # Trips can use official DD/DO.
            # Codes use calculated DD/DO because package DD/DO can be incoherent.
            if pp_category == "TRIP":
                dd_denominator = int(pp["DD"])
                do_denominator = int(pp["DO"])
            else:
                dd_denominator = sum(days_gone_list)
                do_denominator = sum(days_between_trips_list)

            if dd_denominator > 0:
                days_gone_component = (
                    sum(day**2 for day in days_gone_list) / dd_denominator
                )
            else:
                days_gone_component = 0

            if do_denominator > 0:
                days_between_component = (
                    sum(day**2 for day in days_between_trips_list) / do_denominator
                )
            else:
                days_between_component = 0

            blockiness_bonus = (
                0.5 * days_gone_component
                + 0.5 * days_between_component
            )

            pp_scores.append(base_score + blockiness_bonus) # Change this line to multiplication if desired

        if pp_scores:
            line["blockiness_score"] = sum(pp_scores) / len(pp_scores)
        else:
            line["blockiness_score"] = 0

In [16]:
#Tuneable Scoring Blockiness

def capped_weighted_average_block_size(block_lengths, cap):
    """
    Rewards larger blocks, but stops giving extra credit after 'cap'.

    Example:
        cap = 7

        7 days off gets full useful credit.
        14 days off is not treated as twice as good as 7.
    """

    block_lengths = [
        length
        for length in block_lengths
        if length > 0
    ]

    if not block_lengths:
        return 0

    capped_lengths = [
        min(length, cap)
        for length in block_lengths
    ]

    return (
        sum(length ** 2 for length in capped_lengths)
        / sum(capped_lengths)
    )


def merge_touching_work_blocks(work_blocks):
    """
    Merges work blocks that have no real day off between them.

    Example:
        5 on, 0 off, 2 on
        becomes:
        7 on
    """

    if not work_blocks:
        return []

    work_blocks = sorted(
        work_blocks,
        key=lambda block: block["start_date"]
    )

    merged = [work_blocks[0].copy()]

    for block in work_blocks[1:]:
        previous = merged[-1]

        days_off_between = (
            block["start_date"] - previous["end_date"]
        ).days - 1

        if days_off_between <= 0:
            previous["end_date"] = max(
                previous["end_date"],
                block["end_date"]
            )

            previous["days_gone"] = (
                previous["end_date"] - previous["start_date"]
            ).days + 1

        else:
            merged.append(block.copy())

    return merged

from datetime import date, timedelta


def add_blockiness_scores(master_lines, bid_period_info):
    """
    Adds one top-level key to each line:

        line["blockiness_score"]

    Uses bid_period_info instead of bid_start / bid_end.

    bid_period_info format:

        {
            "bid_period_date_range": {
                "start": "2023-05-21",
                "end": "2023-07-16"
            },
            "pay_period_date_ranges": {
                "PP1": {
                    "start": "2023-05-21",
                    "end": "2023-06-17"
                },
                "PP2": {
                    "start": "2023-06-18",
                    "end": "2023-07-15"
                }
            }
        }

    Scoring:
        - Each PP gets its own score.
        - Final line score = average of PP scores.

    Preference order:
        TRIP > VTO > RB > RA > SB > SA > VOR

    Method:
        PP score = category_multiplier * blockiness_bonus
        except VTO, which currently uses a fixed score.
    """

    pay_period_ranges = bid_period_info["pay_period_date_ranges"]

    category_base_scores = {
        "TRIP": 700,
        "VTO": 600,
        "RB": 500,
        "RA": 400,
        "SB": 300,
        "SA": 200,
        "VOR": 100,
        "UNKNOWN": 0,
    }

    code_preference_order = ["VTO", "RB", "RA", "SB", "SA", "VOR"]
    measurable_codes = {"RB", "RA", "SB", "SA", "VOR"}

    for line_number, line in master_lines.items():

        pp_scores = []

        for pp_index, pp in enumerate(line["PPs"]):

            # --------------------------------------------------------
            # 1. Get PP date range from bid_period_info
            # --------------------------------------------------------
            pp_name = pp.get("pp", f"PP{pp_index + 1}")

            if pp_name not in pay_period_ranges:
                pp_scores.append(0)
                continue

            pp_start = date.fromisoformat(pay_period_ranges[pp_name]["start"])
            pp_end = date.fromisoformat(pay_period_ranges[pp_name]["end"])

            trip_blocks = []
            code_dates = {}

            # --------------------------------------------------------
            # 2. Read assignments using your actual master_lines format
            # --------------------------------------------------------
            for assignment in pp["assignments"]:

                # Normal trip assignment
                if "flights" in assignment:

                    start_dates = []
                    end_dates = []

                    for flight in assignment["flights"]:
                        start_dates.append(date.fromisoformat(flight["start_date"]))
                        end_dates.append(date.fromisoformat(flight["end_date"]))

                    trip_start = min(start_dates)
                    trip_end = max(end_dates)

                    trip_blocks.append({
                        "start_date": trip_start,
                        "end_date": trip_end,
                        "days_gone": assignment["total_days_gone"],
                    })

                # Code assignment: {'code': 'VTO', 'date': '2023-05-21'}
                elif "code" in assignment:

                    code = assignment["code"]
                    code_date = date.fromisoformat(assignment["date"])

                    code_dates.setdefault(code, []).append(code_date)

            # --------------------------------------------------------
            # 3. Determine PP category and work blocks
            # --------------------------------------------------------
            if trip_blocks:
                pp_category = "TRIP"
                work_blocks = trip_blocks

            else:
                pp_category = "UNKNOWN"

                for code in code_preference_order:
                    if code in code_dates:
                        pp_category = code
                        break

                work_blocks = []

                # VTO is time off, so it has no work blocks.
                if pp_category == "VTO":
                    pass

                # RB / RA / SB / SA / VOR can be measured as blocks.
                elif pp_category in measurable_codes:

                    all_code_work_dates = []

                    for code, dates in code_dates.items():
                        if code in measurable_codes:
                            all_code_work_dates.extend(dates)

                    all_code_work_dates = sorted(set(all_code_work_dates))

                    # Group consecutive code dates into blocks.
                    if all_code_work_dates:
                        block_start = all_code_work_dates[0]
                        previous_date = all_code_work_dates[0]

                        for current_date in all_code_work_dates[1:]:

                            if current_date == previous_date + timedelta(days=1):
                                previous_date = current_date
                            else:
                                days_gone = (previous_date - block_start).days + 1

                                work_blocks.append({
                                    "start_date": block_start,
                                    "end_date": previous_date,
                                    "days_gone": days_gone,
                                })

                                block_start = current_date
                                previous_date = current_date

                        days_gone = (previous_date - block_start).days + 1

                        work_blocks.append({
                            "start_date": block_start,
                            "end_date": previous_date,
                            "days_gone": days_gone,
                        })

            base_score = category_base_scores.get(pp_category, 0)

            # --------------------------------------------------------
            # 4. Special handling for VTO
            # --------------------------------------------------------
            if pp_category == "VTO":
                # VTO usually covers the whole PP.
                # Treat it as one big clean off block.
                vto_fixed_score = 60

                pp_scores.append(base_score + vto_fixed_score) # Change this line to multiplication if desired
                continue

            # --------------------------------------------------------
            # 5. Calculate improved blockiness bonus
            # --------------------------------------------------------

            work_blocks = merge_touching_work_blocks(work_blocks)
            work_blocks.sort(key=lambda block: block["start_date"])

            if not work_blocks:
                pp_scores.append(base_score)
                continue

            days_gone_list = [
                block["days_gone"]
                for block in work_blocks
            ]

            # --------------------------------------------------------
            # Build off blocks
            # --------------------------------------------------------

            edge_off_gaps = []
            internal_off_gaps = []

            # PP start to first work block
            first_gap = (work_blocks[0]["start_date"] - pp_start).days
            edge_off_gaps.append(max(first_gap, 0))

            # Between work blocks
            for i in range(1, len(work_blocks)):
                previous_end = work_blocks[i - 1]["end_date"]
                next_start = work_blocks[i]["start_date"]

                # Important:
                # If previous_end is Monday and next_start is Wednesday,
                # only Tuesday is off, so subtract 1.
                gap = (next_start - previous_end).days - 1

                internal_off_gaps.append(max(gap, 0))

            # Last work block to PP end
            last_gap = (pp_end - work_blocks[-1]["end_date"]).days
            edge_off_gaps.append(max(last_gap, 0))

            all_off_gaps = edge_off_gaps + internal_off_gaps

            # --------------------------------------------------------
            # Tunable scoring settings
            # --------------------------------------------------------

            ideal_on_block = 7
            ideal_off_block = 7

            on_weight = 40
            off_weight = 40

            min_internal_off_gap = 4
            short_off_penalty_per_day = 10

            min_work_block = 4
            short_work_penalty_per_day = 12

            # --------------------------------------------------------
            # Positive score:
            # reward good-sized blocks, but cap them
            # --------------------------------------------------------

            on_average = capped_weighted_average_block_size(
                days_gone_list,
                cap=ideal_on_block
            )

            off_average = capped_weighted_average_block_size(
                all_off_gaps,
                cap=ideal_off_block
            )

            on_score = (
                on_average / ideal_on_block
            ) * on_weight

            off_score = (
                off_average / ideal_off_block
            ) * off_weight

            # --------------------------------------------------------
            # Penalties:
            # punish ugly 1-day or 2-day internal off gaps
            # --------------------------------------------------------

            short_internal_off_penalty = sum(
                (min_internal_off_gap - gap) * short_off_penalty_per_day
                for gap in internal_off_gaps
                if 0 < gap < min_internal_off_gap
            )

            # Punish isolated tiny work blocks, such as 1 day on
            short_work_block_penalty = sum(
                (min_work_block - days_gone) * short_work_penalty_per_day
                for days_gone in days_gone_list
                if 0 < days_gone < min_work_block
            )

            # Optional: punish too many separate work blocks in one PP
            too_many_blocks_penalty = max(
                0,
                len(work_blocks) - 3
            ) * 5

            total_penalty = (
                short_internal_off_penalty
                + short_work_block_penalty
                + too_many_blocks_penalty
            )

            blockiness_bonus = (
                on_score
                + off_score
                - total_penalty
            )

            # Keep bonus inside a controlled range.
            # This keeps the category base score dominant.
            blockiness_bonus = max(0, min(blockiness_bonus, 99))

            pp_scores.append(base_score + blockiness_bonus)
        
        if pp_scores:
            line["blockiness_score"] = sum(pp_scores) / len(pp_scores)
        else:
            line["blockiness_score"] = 0

In [14]:
#Harmonic Average Blockiness

def weighted_block_average(lengths):
    """
    Rewards large blocks.

    Example:
        [14, 14] scores high.
        [15, 1, 1, 14] also scores high by itself,
        which is why we also need harmonic_block_average().
    """

    lengths = [
        length
        for length in lengths
        if length > 0
    ]

    if not lengths:
        return 0

    return sum(length ** 2 for length in lengths) / sum(lengths)


def harmonic_block_average(lengths):
    """
    Punishes tiny blocks naturally.

    A single 1-day block drags the result down heavily.
    """

    lengths = [
        length
        for length in lengths
        if length > 0
    ]

    if not lengths:
        return 0

    return len(lengths) / sum(1 / length for length in lengths)


def block_quality(lengths):
    """
    Combines:
        - reward for large blocks
        - punishment for tiny blocks

    No ideal block length is used.
    """

    lengths = [
        length
        for length in lengths
        if length > 0
    ]

    if not lengths:
        return 0

    large_block_score = weighted_block_average(lengths)
    small_block_score = harmonic_block_average(lengths)

    return (large_block_score * small_block_score) ** 0.5


def off_block_quality(all_off_gaps, internal_off_gaps):
    """
    Gives credit for large off blocks,
    but uses internal off gaps to detect ugly 1-day gaps between work blocks.

    Small edge gaps at the very beginning/end of the PP are less important
    because they may connect to another PP outside this scoring window.
    """

    all_off_gaps = [
        gap
        for gap in all_off_gaps
        if gap > 0
    ]

    internal_off_gaps = [
        gap
        for gap in internal_off_gaps
        if gap > 0
    ]

    if not all_off_gaps:
        return 0

    large_off_score = weighted_block_average(all_off_gaps)

    if internal_off_gaps:
        small_gap_score = harmonic_block_average(internal_off_gaps)
    else:
        small_gap_score = harmonic_block_average(all_off_gaps)

    return (large_off_score * small_gap_score) ** 0.5

def add_blockiness_scores(master_lines, bid_period_info):
    """
    Adds one top-level key to each line:

        line["blockiness_score"]

    Uses bid_period_info instead of bid_start / bid_end.

    bid_period_info format:

        {
            "bid_period_date_range": {
                "start": "2023-05-21",
                "end": "2023-07-16"
            },
            "pay_period_date_ranges": {
                "PP1": {
                    "start": "2023-05-21",
                    "end": "2023-06-17"
                },
                "PP2": {
                    "start": "2023-06-18",
                    "end": "2023-07-15"
                }
            }
        }

    Scoring:
        - Each PP gets its own score.
        - Final line score = average of PP scores.

    Preference order:
        TRIP > VTO > RB > RA > SB > SA > VOR

    Method:
        PP score = category_multiplier * blockiness_bonus
        except VTO, which currently uses a fixed score.
    """

    pay_period_ranges = bid_period_info["pay_period_date_ranges"]

    category_base_scores = {
        "TRIP": 700,
        "VTO": 600,
        "RB": 500,
        "RA": 400,
        "SB": 300,
        "SA": 200,
        "VOR": 100,
        "UNKNOWN": 0,
    }

    code_preference_order = ["VTO", "RB", "RA", "SB", "SA", "VOR"]
    measurable_codes = {"RB", "RA", "SB", "SA", "VOR"}

    for line_number, line in master_lines.items():

        pp_scores = []

        for pp_index, pp in enumerate(line["PPs"]):

            # --------------------------------------------------------
            # 1. Get PP date range from bid_period_info
            # --------------------------------------------------------
            pp_name = pp.get("pp", f"PP{pp_index + 1}")

            if pp_name not in pay_period_ranges:
                pp_scores.append(0)
                continue

            pp_start = date.fromisoformat(pay_period_ranges[pp_name]["start"])
            pp_end = date.fromisoformat(pay_period_ranges[pp_name]["end"])

            trip_blocks = []
            code_dates = {}

            # --------------------------------------------------------
            # 2. Read assignments using your actual master_lines format
            # --------------------------------------------------------
            for assignment in pp["assignments"]:

                # Normal trip assignment
                if "flights" in assignment:

                    start_dates = []
                    end_dates = []

                    for flight in assignment["flights"]:
                        start_dates.append(date.fromisoformat(flight["start_date"]))
                        end_dates.append(date.fromisoformat(flight["end_date"]))

                    trip_start = min(start_dates)
                    trip_end = max(end_dates)

                    trip_blocks.append({
                        "start_date": trip_start,
                        "end_date": trip_end,
                        "days_gone": assignment["total_days_gone"],
                    })

                # Code assignment: {'code': 'VTO', 'date': '2023-05-21'}
                elif "code" in assignment:

                    code = assignment["code"]
                    code_date = date.fromisoformat(assignment["date"])

                    code_dates.setdefault(code, []).append(code_date)

            # --------------------------------------------------------
            # 3. Determine PP category and work blocks
            # --------------------------------------------------------
            if trip_blocks:
                pp_category = "TRIP"
                work_blocks = trip_blocks

            else:
                pp_category = "UNKNOWN"

                for code in code_preference_order:
                    if code in code_dates:
                        pp_category = code
                        break

                work_blocks = []

                # VTO is time off, so it has no work blocks.
                if pp_category == "VTO":
                    pass

                # RB / RA / SB / SA / VOR can be measured as blocks.
                elif pp_category in measurable_codes:

                    all_code_work_dates = []

                    for code, dates in code_dates.items():
                        if code in measurable_codes:
                            all_code_work_dates.extend(dates)

                    all_code_work_dates = sorted(set(all_code_work_dates))

                    # Group consecutive code dates into blocks.
                    if all_code_work_dates:
                        block_start = all_code_work_dates[0]
                        previous_date = all_code_work_dates[0]

                        for current_date in all_code_work_dates[1:]:

                            if current_date == previous_date + timedelta(days=1):
                                previous_date = current_date
                            else:
                                days_gone = (previous_date - block_start).days + 1

                                work_blocks.append({
                                    "start_date": block_start,
                                    "end_date": previous_date,
                                    "days_gone": days_gone,
                                })

                                block_start = current_date
                                previous_date = current_date

                        days_gone = (previous_date - block_start).days + 1

                        work_blocks.append({
                            "start_date": block_start,
                            "end_date": previous_date,
                            "days_gone": days_gone,
                        })

            base_score = category_base_scores.get(pp_category, 0)

            # --------------------------------------------------------
            # 4. Special handling for VTO
            # --------------------------------------------------------
            if pp_category == "VTO":
                # VTO usually covers the whole PP.
                # Treat it as one big clean off block.
                vto_fixed_score = 60

                pp_scores.append(base_score + vto_fixed_score) # Change this line to multiplication if desired
                continue

            # --------------------------------------------------------
            # 5. Calculate blockiness bonus
            # --------------------------------------------------------

            work_blocks.sort(key=lambda block: block["start_date"])

            if not work_blocks:
                pp_scores.append(base_score)
                continue

            days_gone_list = [
                block["days_gone"]
                for block in work_blocks
            ]

            edge_off_gaps = []
            internal_off_gaps = []

            # PP start to first work block
            first_gap = (work_blocks[0]["start_date"] - pp_start).days
            edge_off_gaps.append(max(first_gap, 0))

            # Between work blocks
            for i in range(1, len(work_blocks)):
                previous_end = work_blocks[i - 1]["end_date"]
                next_start = work_blocks[i]["start_date"]

                # Important:
                # If previous_end is Monday and next_start is Wednesday,
                # there is only 1 day off: Tuesday.
                gap = (next_start - previous_end).days - 1

                internal_off_gaps.append(max(gap, 0))

            # Last work block to PP end
            last_gap = (pp_end - work_blocks[-1]["end_date"]).days
            edge_off_gaps.append(max(last_gap, 0))

            all_off_gaps = edge_off_gaps + internal_off_gaps

            work_quality = block_quality(days_gone_list)

            off_quality = off_block_quality(
                all_off_gaps=all_off_gaps,
                internal_off_gaps=internal_off_gaps,
            )

            blockiness_bonus = (
                work_quality
                + off_quality
            ) / 2

            pp_scores.append(base_score + blockiness_bonus)
        
        if pp_scores:
            line["blockiness_score"] = sum(pp_scores) / len(pp_scores)
        else:
            line["blockiness_score"] = 0

In [11]:
#Red flag blockiness

from datetime import date, timedelta


def weighted_block_average(lengths):
    """
    Rewards larger blocks.
    A 14-day block is better than two 7-day blocks.
    No ideal block length is used.
    """
    lengths = [length for length in lengths if length > 0]

    if not lengths:
        return 0

    return sum(length ** 2 for length in lengths) / sum(lengths)


def harmonic_block_average(lengths):
    """
    Punishes tiny blocks.
    A single 1-day block drags this down hard.
    """
    lengths = [length for length in lengths if length > 0]

    if not lengths:
        return 0

    return len(lengths) / sum(1 / length for length in lengths)


def block_quality(lengths):
    """
    Combines:
        - reward for big blocks
        - punishment for tiny blocks

    No ideal block length is used.
    """
    lengths = [length for length in lengths if length > 0]

    if not lengths:
        return 0

    weighted = weighted_block_average(lengths)
    harmonic = harmonic_block_average(lengths)

    return (weighted * harmonic) ** 0.5


def merge_touching_work_blocks(work_blocks):
    """
    Merges work blocks that have no real day off between them.

    Example:
        5 on, 0 off, 2 on
        becomes:
        7 on
    """
    if not work_blocks:
        return []

    work_blocks = sorted(work_blocks, key=lambda block: block["start_date"])

    merged = [work_blocks[0].copy()]

    for block in work_blocks[1:]:
        previous = merged[-1]

        days_off_between = (
            block["start_date"] - previous["end_date"]
        ).days - 1

        if days_off_between <= 0:
            previous["end_date"] = max(
                previous["end_date"],
                block["end_date"]
            )

            previous["days_gone"] = (
                previous["end_date"] - previous["start_date"]
            ).days + 1

        else:
            merged.append(block.copy())

    return merged


def calculate_red_flag_penalty(
    work_blocks,
    edge_off_gaps,
    internal_off_gaps,
):
    """
    Penalizes the things that usually make a line feel ugly manually:

        - 1-day or 2-day work islands
        - 1-day or 2-day off gaps between work blocks
        - too many separate work blocks
        - a short work block surrounded by large off blocks

    These are not ideal block lengths.
    They are anti-choppiness penalties.
    """

    penalty = 0

    work_lengths = [
        block["days_gone"]
        for block in work_blocks
        if block["days_gone"] > 0
    ]

    # ------------------------------------------------------------
    # 1. Penalize short work blocks
    # ------------------------------------------------------------
    # 1-day work block = big penalty
    # 2-day work block = medium penalty
    # 3-day work block = small penalty
    for length in work_lengths:
        if length == 1:
            penalty += 28
        elif length == 2:
            penalty += 18
        elif length == 3:
            penalty += 8

    # ------------------------------------------------------------
    # 2. Penalize short internal off gaps
    # ------------------------------------------------------------
    # These are days off between work blocks.
    # Edge days off are not punished here because they may connect
    # to another pay period.
    for gap in internal_off_gaps:
        if gap == 1:
            penalty += 24
        elif gap == 2:
            penalty += 14
        elif gap == 3:
            penalty += 6

    # ------------------------------------------------------------
    # 3. Penalize too many separate work blocks
    # ------------------------------------------------------------
    # One or two work blocks in a PP can still be clean.
    # Three starts to feel chopped up.
    # Four or more is usually ugly.
    if len(work_blocks) > 2:
        penalty += (len(work_blocks) - 2) * 10

    # ------------------------------------------------------------
    # 4. Penalize isolated work islands inside off time
    # ------------------------------------------------------------
    # Example:
    #     10 off, 1 on, 12 off
    #
    # This is worse than simply having a 1-day work block.
    # It breaks what would otherwise be a large off block.
    for i, block in enumerate(work_blocks):
        length = block["days_gone"]

        left_off = None
        right_off = None

        if i == 0:
            left_off = edge_off_gaps[0]
        else:
            left_off = internal_off_gaps[i - 1]

        if i == len(work_blocks) - 1:
            right_off = edge_off_gaps[-1]
        else:
            right_off = internal_off_gaps[i]

        if left_off is None or right_off is None:
            continue

        surrounding_off = left_off + right_off

        if surrounding_off >= 7:
            if length == 1:
                penalty += 30
            elif length == 2:
                penalty += 20
            elif length == 3:
                penalty += 10

    return penalty


def safe_harmonic_average(values):
    """
    Harmonic average for PP bonuses.

    This makes one ugly PP drag the final line score down more
    than a normal average would.
    """
    values = [value for value in values if value > 0]

    if not values:
        return 0

    return len(values) / sum(1 / value for value in values)


def add_blockiness_scores(master_lines, bid_period_info):
    """
    Adds:

        line["blockiness_score"]

    Main idea:
        category base score + clean block bonus

    This version avoids ideal 7-on / 7-off tuning.

    Instead it:
        - rewards large work blocks
        - rewards large off blocks
        - punishes tiny work islands
        - punishes tiny internal off gaps
        - punishes too many separate work blocks
        - lets one ugly PP drag down the final line score
    """

    pay_period_ranges = bid_period_info["pay_period_date_ranges"]

    category_base_scores = {
        "TRIP": 700,
        "VTO": 600,
        "RB": 500,
        "RA": 400,
        "SB": 300,
        "SA": 200,
        "VOR": 100,
        "UNKNOWN": 0,
    }

    code_preference_order = ["VTO", "RB", "RA", "SB", "SA", "VOR"]
    measurable_codes = {"RB", "RA", "SB", "SA", "VOR"}

    for line_number, line in master_lines.items():

        pp_base_scores = []
        pp_block_bonuses = []

        for pp_index, pp in enumerate(line["PPs"]):

            pp_name = pp.get("pp", f"PP{pp_index + 1}")

            if pp_name not in pay_period_ranges:
                pp_base_scores.append(0)
                pp_block_bonuses.append(0)
                continue

            pp_start = date.fromisoformat(pay_period_ranges[pp_name]["start"])
            pp_end = date.fromisoformat(pay_period_ranges[pp_name]["end"])

            trip_blocks = []
            code_dates = {}

            # --------------------------------------------------------
            # Read assignments
            # --------------------------------------------------------
            for assignment in pp["assignments"]:

                if "flights" in assignment:

                    start_dates = []
                    end_dates = []

                    for flight in assignment["flights"]:
                        start_dates.append(date.fromisoformat(flight["start_date"]))
                        end_dates.append(date.fromisoformat(flight["end_date"]))

                    trip_start = min(start_dates)
                    trip_end = max(end_dates)

                    trip_blocks.append({
                        "start_date": trip_start,
                        "end_date": trip_end,
                        "days_gone": (trip_end - trip_start).days + 1,
                    })

                elif "code" in assignment:

                    code = assignment["code"]
                    code_date = date.fromisoformat(assignment["date"])

                    code_dates.setdefault(code, []).append(code_date)

            # --------------------------------------------------------
            # Determine PP category and work blocks
            # --------------------------------------------------------
            if trip_blocks:
                pp_category = "TRIP"
                work_blocks = trip_blocks

            else:
                pp_category = "UNKNOWN"

                for code in code_preference_order:
                    if code in code_dates:
                        pp_category = code
                        break

                work_blocks = []

                if pp_category == "VTO":
                    pass

                elif pp_category in measurable_codes:

                    all_code_work_dates = []

                    for code, dates in code_dates.items():
                        if code in measurable_codes:
                            all_code_work_dates.extend(dates)

                    all_code_work_dates = sorted(set(all_code_work_dates))

                    if all_code_work_dates:
                        block_start = all_code_work_dates[0]
                        previous_date = all_code_work_dates[0]

                        for current_date in all_code_work_dates[1:]:

                            if current_date == previous_date + timedelta(days=1):
                                previous_date = current_date
                            else:
                                work_blocks.append({
                                    "start_date": block_start,
                                    "end_date": previous_date,
                                    "days_gone": (previous_date - block_start).days + 1,
                                })

                                block_start = current_date
                                previous_date = current_date

                        work_blocks.append({
                            "start_date": block_start,
                            "end_date": previous_date,
                            "days_gone": (previous_date - block_start).days + 1,
                        })

            base_score = category_base_scores.get(pp_category, 0)
            pp_base_scores.append(base_score)

            # --------------------------------------------------------
            # VTO handling
            # --------------------------------------------------------
            if pp_category == "VTO":
                # VTO is clean time off.
                # Keep the bonus below 100 so VTO cannot outrank TRIP
                # only because of blockiness.
                pp_block_bonuses.append(95)
                continue

            # --------------------------------------------------------
            # No work blocks
            # --------------------------------------------------------
            work_blocks = merge_touching_work_blocks(work_blocks)
            work_blocks.sort(key=lambda block: block["start_date"])

            if not work_blocks:
                pp_block_bonuses.append(0)
                continue

            # --------------------------------------------------------
            # Build off gaps
            # --------------------------------------------------------
            edge_off_gaps = []
            internal_off_gaps = []

            first_gap = (work_blocks[0]["start_date"] - pp_start).days
            edge_off_gaps.append(max(first_gap, 0))

            for i in range(1, len(work_blocks)):
                previous_end = work_blocks[i - 1]["end_date"]
                next_start = work_blocks[i]["start_date"]

                gap = (next_start - previous_end).days - 1
                internal_off_gaps.append(max(gap, 0))

            last_gap = (pp_end - work_blocks[-1]["end_date"]).days
            edge_off_gaps.append(max(last_gap, 0))

            all_off_gaps = edge_off_gaps + internal_off_gaps

            work_lengths = [
                block["days_gone"]
                for block in work_blocks
                if block["days_gone"] > 0
            ]

            off_lengths = [
                gap
                for gap in all_off_gaps
                if gap > 0
            ]

            # --------------------------------------------------------
            # Positive block quality
            # --------------------------------------------------------
            work_quality = block_quality(work_lengths)
            off_quality = block_quality(off_lengths)

            raw_bonus = (
                0.50 * work_quality
                + 0.50 * off_quality
            )

            # Scale the natural block size into a 0-99 bonus range.
            # No ideal 7-day target is used.
            raw_bonus = raw_bonus * 7

            # --------------------------------------------------------
            # Red-flag penalties
            # --------------------------------------------------------
            penalty = calculate_red_flag_penalty(
                work_blocks=work_blocks,
                edge_off_gaps=edge_off_gaps,
                internal_off_gaps=internal_off_gaps,
            )

            final_bonus = raw_bonus - penalty

            # Keep category base dominant.
            final_bonus = max(0, min(final_bonus, 99))

            pp_block_bonuses.append(final_bonus)

        # ------------------------------------------------------------
        # Final line score
        # ------------------------------------------------------------
        if pp_base_scores:
            average_base_score = sum(pp_base_scores) / len(pp_base_scores)
        else:
            average_base_score = 0

        if pp_block_bonuses:
            average_bonus = sum(pp_block_bonuses) / len(pp_block_bonuses)
            harmonic_bonus = safe_harmonic_average(pp_block_bonuses)

            # This is important:
            # Do not let one excellent PP completely hide one ugly PP.
            final_bonus = (
                0.60 * average_bonus
                + 0.40 * harmonic_bonus
            )
        else:
            final_bonus = 0

        line["blockiness_score"] = average_base_score + final_bonus

In [12]:
from itertools import combinations


def get_blockiness_ranking(master_lines, score_key="blockiness_score"):
    """
    Returns line numbers ranked from best blockiness score to worst.
    """
    return [
        line_number
        for line_number, line_data in sorted(
            master_lines.items(),
            key=lambda item: item[1].get(score_key, 0),
            reverse=True
        )
    ]


def compare_rankings(calculated_order, reference_order):
    """
    Compares two rankings.

    calculated_order:
        The order produced by your blockiness_score function.

    reference_order:
        Your own preferred/manual ranking.

    Returns a dictionary with similarity information.
    """

    # Convert line numbers to strings so 12 and "12" match
    calculated_order = [str(x).strip() for x in calculated_order]
    reference_order = [str(x).strip() for x in reference_order]

    # Remove duplicates while preserving order
    calculated_order = list(dict.fromkeys(calculated_order))
    reference_order = list(dict.fromkeys(reference_order))

    calculated_set = set(calculated_order)
    reference_set = set(reference_order)

    common_lines = [x for x in reference_order if x in calculated_set]

    missing_from_calculated = [
        x for x in reference_order
        if x not in calculated_set
    ]

    if len(common_lines) < 2:
        return {
            "error": "Need at least 2 matching line numbers to compare rankings.",
            "common_lines": common_lines,
            "missing_from_calculated": missing_from_calculated,
        }

    # Re-rank only the common lines
    calculated_common_order = [
        x for x in calculated_order
        if x in set(common_lines)
    ]

    reference_common_order = [
        x for x in reference_order
        if x in set(common_lines)
    ]

    calculated_rank = {
        line_number: rank
        for rank, line_number in enumerate(calculated_common_order, start=1)
    }

    reference_rank = {
        line_number: rank
        for rank, line_number in enumerate(reference_common_order, start=1)
    }

    position_differences = {
        line_number: abs(calculated_rank[line_number] - reference_rank[line_number])
        for line_number in common_lines
    }

    average_position_difference = (
        sum(position_differences.values()) / len(position_differences)
    )

    exact_position_matches = sum(
        1
        for line_number in common_lines
        if calculated_rank[line_number] == reference_rank[line_number]
    )

    # Pairwise comparison:
    # If your list says 9 is better than 8,
    # does blockiness also say 9 is better than 8?
    agreements = 0
    disagreements = 0

    for line_a, line_b in combinations(common_lines, 2):
        reference_prefers_a = reference_rank[line_a] < reference_rank[line_b]
        calculated_prefers_a = calculated_rank[line_a] < calculated_rank[line_b]

        if reference_prefers_a == calculated_prefers_a:
            agreements += 1
        else:
            disagreements += 1

    total_pairs = agreements + disagreements

    pairwise_agreement_percent = agreements / total_pairs * 100

    # Spearman footrule similarity, normalized from 0% to 100%
    footrule_distance = sum(position_differences.values())
    n = len(common_lines)
    max_footrule_distance = (n * n) // 2

    if max_footrule_distance == 0:
        position_similarity_percent = 100
    else:
        position_similarity_percent = (
            1 - footrule_distance / max_footrule_distance
        ) * 100

    return {
        "lines_compared": len(common_lines),
        "missing_from_calculated": missing_from_calculated,
        "pairwise_agreement_percent": pairwise_agreement_percent,
        "position_similarity_percent": position_similarity_percent,
        "average_position_difference": average_position_difference,
        "exact_position_matches": exact_position_matches,
        "comparison": [
            {
                "line_number": line_number,
                "reference_rank": reference_rank[line_number],
                "calculated_rank": calculated_rank[line_number],
                "position_difference": position_differences[line_number],
            }
            for line_number in reference_common_order
        ],
    }


def print_ranking_comparison(result):
    """
    Pretty prints the result from compare_rankings().
    """

    if "error" in result:
        print(result["error"])
        print("Common lines:", result["common_lines"])
        print("Missing lines:", result["missing_from_calculated"])
        return

    print()
    print("Ranking comparison")
    print("-" * 50)

    print(f"Lines compared:              {result['lines_compared']}")
    print(f"Pairwise agreement:          {result['pairwise_agreement_percent']:.1f}%")
    print(f"Position similarity:         {result['position_similarity_percent']:.1f}%")
    print(f"Average position difference: {result['average_position_difference']:.2f}")
    print(f"Exact position matches:      {result['exact_position_matches']}")

    if result["missing_from_calculated"]:
        print()
        print("Missing from calculated ranking:")
        print(result["missing_from_calculated"])

    print()
    print("line_number  |  reference_rank  |  calculated_rank  |  difference")
    print("-" * 65)

    for row in result["comparison"]:
        print(
            f"{row['line_number']:<12} |  "
            f"{row['reference_rank']:<14} |  "
            f"{row['calculated_rank']:<15} |  "
            f"{row['position_difference']}"
        )

In [19]:
add_blockiness_scores(master_lines, bid_period_info)
blockiness_order = get_blockiness_ranking(master_lines)

my_reference_order2602 = [25,26,15,13,9,16,10,11,12,14,23,27,28,32,33,24,29,30,31]
my_reference_order2603 = [13,6,7,10,12,21,31,32,34,25,26,28,29,8,9,23,24,27,11,14,30,33,15,20,22]
my_reference_order2304 = [51,52,54,44,41,47,49,37,48,45,102,73,103,43,24,31,23,27,56,38,30,35,39,33,40,26,28,34,36,22,42,50,32,25,111]
my_reference_order2507 = [22,21,29,25,20,28,24,26,23,37,25,27,32,45,30,31,33,34,36,47,67,65,38,44,39,46,63]

result = compare_rankings(
    calculated_order=blockiness_order,
    reference_order=my_reference_order2507,
)

print_ranking_comparison(result)


Ranking comparison
--------------------------------------------------
Lines compared:              26
Pairwise agreement:          70.2%
Position similarity:         60.4%
Average position difference: 5.15
Exact position matches:      2

line_number  |  reference_rank  |  calculated_rank  |  difference
-----------------------------------------------------------------
22           |  1              |  10              |  9
21           |  2              |  8               |  6
29           |  3              |  16              |  13
25           |  4              |  7               |  3
20           |  5              |  13              |  8
28           |  6              |  5               |  1
24           |  7              |  11              |  4
26           |  8              |  12              |  4
23           |  9              |  6               |  3
37           |  10             |  3               |  7
27           |  11             |  4               |  7
32           |  12     

In [ ]:
add_blockiness_scores(master_lines, bid_period_info)

print("line_number  |  blockiness_score")
print("-" * 34)

for line_number, line_data in sorted(
    master_lines.items(),
    key=lambda item: item[1].get("blockiness_score", 0),
    reverse=True
):
    print(f"{line_number:<12} |  {line_data.get('blockiness_score', 0)}")

In [ ]:
import re


def add_company_ticket_percentages(master_lines):
    """
    Adds company-paid ticket percentage to each pay period and each line.

    Logic:
        - Look only at the first and last flight of each trip.
        - If first flight has DH + airline, except DH UPS, count as company-paid ticket to work.
        - If last flight has DH + airline, except DH UPS, count as company-paid ticket from work.
        - Non-trip assignments like VTO, VOR, RA, RB, SA, SB are ignored.

    Adds to each line:
        line["company_ticket_pct"]
    """

    dh_pattern = re.compile(r"\bDH\s+([A-Z0-9]+)\b", re.IGNORECASE)

    for line_num, line in master_lines.items():

        line_to_work = 0
        line_from_work = 0
        line_ticket_count = 0
        line_ticket_possible = 0

        for pp in line.get("PPs", []):

            pp_to_work = 0
            pp_from_work = 0
            pp_ticket_count = 0
            pp_ticket_possible = 0

            for assignment in pp.get("assignments", []):

                flights = assignment.get("flights")

                # Skip VTO / VOR / RA / RB / SA / SB / anything that is not a trip
                if not flights:
                    continue

                first_flight = flights[0]
                last_flight = flights[-1]

                # Each trip has 2 possible ticket positions:
                #   1. ticket to work
                #   2. ticket from work
                pp_ticket_possible += 2

                # -------------------------
                # Check first flight
                # -------------------------
                first_flags = first_flight.get("route_flags") or []

                if isinstance(first_flags, str):
                    first_flags = [first_flags]

                first_has_company_ticket = False

                for flag in first_flags:
                    flag = str(flag).upper()
                    matches = dh_pattern.findall(flag)

                    for carrier in matches:
                        if carrier.upper() != "UPS":
                            first_has_company_ticket = True
                            break

                    if first_has_company_ticket:
                        break

                if first_has_company_ticket:
                    pp_to_work += 1
                    pp_ticket_count += 1

                # -------------------------
                # Check last flight
                # -------------------------
                last_flags = last_flight.get("route_flags") or []

                if isinstance(last_flags, str):
                    last_flags = [last_flags]

                last_has_company_ticket = False

                for flag in last_flags:
                    flag = str(flag).upper()
                    matches = dh_pattern.findall(flag)

                    for carrier in matches:
                        if carrier.upper() != "UPS":
                            last_has_company_ticket = True
                            break

                    if last_has_company_ticket:
                        break

                if last_has_company_ticket:
                    pp_from_work += 1
                    pp_ticket_count += 1

            if pp_ticket_possible:
                pp_ticket_pct = round((pp_ticket_count / pp_ticket_possible) * 100, 1)
            else:
                pp_ticket_pct = 0.0

            line_to_work += pp_to_work
            line_from_work += pp_from_work
            line_ticket_count += pp_ticket_count
            line_ticket_possible += pp_ticket_possible

        if line_ticket_possible:
            line_ticket_pct = round((line_ticket_count / line_ticket_possible) * 100, 1)
        else:
            line_ticket_pct = 0.0
        

        line["company_ticket_pct"] = line_ticket_pct

add_company_ticket_percentages(master_lines)

In [ ]:
pprint(master_lines)

In [ ]:
from datetime import date, datetime, timedelta


def add_training_fit_score(
    master_lines,
    training_start,
    training_end,
    bid_period_info,
    trip_weight=0.80,
    off_edge_weight=0.20,
):
    """
    Adds a training fit score to each line.

    Higher score = better.

    Main idea:
        1. Reward training dates that fall on trip days.
        2. Penalize training dates that fall in the middle of days-off blocks.
        3. Prefer training that either replaces work or touches the edge of an off block.

    Date logic:
        Uses inclusive calendar dates.

        Example:
            training_start = "2023-06-01"
            training_end   = "2023-06-03"

        Means:
            Jun 01, Jun 02, Jun 03

    Adds:
        line["training_fit_score"]

    Optional debug fields if save_details=True:
        line["training_trip_overlap_pct"]
        line["training_off_middle_penalty_pct"]
    """

    def to_date(value):
        if isinstance(value, datetime):
            return value.date()

        if isinstance(value, date):
            return value

        if isinstance(value, str):
            return date.fromisoformat(value)

        raise TypeError(f"Unsupported date value: {value!r}")

    def date_range_inclusive(start, end):
        current = start
        while current <= end:
            yield current
            current += timedelta(days=1)

    def build_off_blocks(bid_days, trip_days):
        off_blocks = []

        current_start = None
        previous_day = None

        for day in sorted(bid_days):
            is_off_day = day not in trip_days

            if is_off_day:
                if current_start is None:
                    current_start = day

                previous_day = day

            else:
                if current_start is not None:
                    off_blocks.append((current_start, previous_day))
                    current_start = None
                    previous_day = None

        if current_start is not None:
            off_blocks.append((current_start, previous_day))

        return off_blocks

    def get_trip_days_for_line(line):
        trip_days = set()

        for pp in line.get("PPs", []):
            for assignment in pp.get("assignments", []):

                flights = assignment.get("flights")

                # Skip VTO, VOR, RA, RB, SA, SB, etc.
                if not flights:
                    continue

                start_dates = [
                    to_date(flight["start_date"])
                    for flight in flights
                    if flight.get("start_date")
                ]

                end_dates = [
                    to_date(flight["end_date"])
                    for flight in flights
                    if flight.get("end_date")
                ]

                if not start_dates or not end_dates:
                    continue

                trip_start = min(start_dates)
                trip_end = max(end_dates)

                for day in date_range_inclusive(trip_start, trip_end):
                    trip_days.add(day)

        return trip_days

    training_start = to_date(training_start)
    training_end = to_date(training_end)
    bid_start = to_date(bid_period_info['bid_period_date_range']['start'])
    bid_end = to_date(bid_period_info['bid_period_date_range']['end'])

    if training_end < training_start:
        raise ValueError("training_end must be on or after training_start")

    if bid_end < bid_start:
        raise ValueError("bid_end must be on or after bid_start")

    training_days = list(date_range_inclusive(training_start, training_end))
    bid_days = set(date_range_inclusive(bid_start, bid_end))

    training_total_days = len(training_days)

    for line_num, line in master_lines.items():

        trip_days = get_trip_days_for_line(line)

        # 1. Reward training that overlaps trips
        training_trip_days = [
            day for day in training_days
            if day in trip_days
        ]

        trip_overlap_pct = (
            len(training_trip_days) / training_total_days
        ) * 100

        # 2. Look at training days that fall on days off
        training_off_days = [
            day for day in training_days
            if day not in trip_days
        ]

        off_blocks = build_off_blocks(bid_days, trip_days)

        off_day_to_block = {}

        for block_start, block_end in off_blocks:
            for day in date_range_inclusive(block_start, block_end):
                off_day_to_block[day] = (block_start, block_end)

        middle_penalty_values = []

        for day in training_off_days:
            block = off_day_to_block.get(day)

            # If the training day is outside the bid period,
            # treat it as a bad off-day placement.
            if block is None:
                middle_penalty_values.append(1.0)
                continue

            block_start, block_end = block
            block_length = (block_end - block_start).days + 1

            if block_length <= 1:
                middle_penalty_values.append(0.0)
                continue

            days_from_left_edge = (day - block_start).days
            days_from_right_edge = (block_end - day).days

            edge_distance = min(days_from_left_edge, days_from_right_edge)

            max_possible_edge_distance = (block_length - 1) / 2

            middle_penalty = edge_distance / max_possible_edge_distance

            middle_penalty_values.append(middle_penalty)

        if middle_penalty_values:
            off_middle_penalty_pct = (
                sum(middle_penalty_values) / len(middle_penalty_values)
            ) * 100
        else:
            off_middle_penalty_pct = 0.0

        off_edge_score = 100 - off_middle_penalty_pct

        training_fit_score = (
            trip_weight * trip_overlap_pct
            + off_edge_weight * off_edge_score
        )

        line["training_fit_score"] = round(training_fit_score, 1)

add_training_fit_score(master_lines, "2023-07-06", "2023-07-10",bid_period_info)

In [ ]:
pprint(master_lines)

In [ ]:
from datetime import date, datetime, timedelta


def add_training_fit_score(
    master_lines,
    training_start,
    training_end,
    bid_start,
    bid_end,
    trip_weight=0.80,
    off_edge_weight=0.20,
    reserve_work_codes=None,
):
    """
    Adds a training fit score to each line.

    Higher score = better.

    Main idea:
        1. Reward training dates that fall on trip days or reserve/on-call days.
        2. Penalize training dates that fall in the middle of true days-off blocks.
        3. Prefer training that either replaces work/reserve or touches the edge of an off block.

    Date logic:
        Uses inclusive calendar dates.

        Example:
            training_start = "2023-06-01"
            training_end   = "2023-06-03"

        Means:
            Jun 01, Jun 02, Jun 03

    Adds:
        line["training_fit_score"]
    """

    if reserve_work_codes is None:
        reserve_work_codes = {"RA", "RB", "SA", "SB"}

    def to_date(value):
        if isinstance(value, datetime):
            return value.date()

        if isinstance(value, date):
            return value

        if isinstance(value, str):
            return date.fromisoformat(value)

        raise TypeError(f"Unsupported date value: {value!r}")

    def date_range_inclusive(start, end):
        current = start
        while current <= end:
            yield current
            current += timedelta(days=1)

    def build_off_blocks(bid_days, work_days):
        off_blocks = []

        current_start = None
        previous_day = None

        for day in sorted(bid_days):
            is_off_day = day not in work_days

            if is_off_day:
                if current_start is None:
                    current_start = day

                previous_day = day

            else:
                if current_start is not None:
                    off_blocks.append((current_start, previous_day))
                    current_start = None
                    previous_day = None

        if current_start is not None:
            off_blocks.append((current_start, previous_day))

        return off_blocks

    def get_work_days_for_line(line):
        """
        Returns all days that should count as 'on' days.

        Includes:
            - actual trip days from flight start/end dates
            - RA/RB/SA/SB reserve/on-call days from assignment code/date

        Excludes:
            - VTO
            - VOR, unless you later decide to include it
            - any other non-work code
        """

        work_days = set()

        for pp in line.get("PPs", []):
            for assignment in pp.get("assignments", []):

                flights = assignment.get("flights")

                # Case 1: real trip assignment
                if flights:
                    start_dates = [
                        to_date(flight["start_date"])
                        for flight in flights
                        if flight.get("start_date")
                    ]

                    end_dates = [
                        to_date(flight["end_date"])
                        for flight in flights
                        if flight.get("end_date")
                    ]

                    if not start_dates or not end_dates:
                        continue

                    trip_start = min(start_dates)
                    trip_end = max(end_dates)

                    for day in date_range_inclusive(trip_start, trip_end):
                        work_days.add(day)

                    continue

                # Case 2: reserve/on-call assignment with code/date
                code = assignment.get("code")
                assignment_date = assignment.get("date")

                if code in reserve_work_codes and assignment_date:
                    work_days.add(to_date(assignment_date))

        return work_days

    training_start = to_date(training_start)
    training_end = to_date(training_end)
    bid_start = to_date(bid_start)
    bid_end = to_date(bid_end)

    if training_end < training_start:
        raise ValueError("training_end must be on or after training_start")

    if bid_end < bid_start:
        raise ValueError("bid_end must be on or after bid_start")

    training_days = list(date_range_inclusive(training_start, training_end))
    bid_days = set(date_range_inclusive(bid_start, bid_end))

    training_total_days = len(training_days)

    for line_num, line in master_lines.items():

        work_days = get_work_days_for_line(line)

        # 1. Reward training that overlaps trips or reserve/on-call days
        training_work_days = [
            day for day in training_days
            if day in work_days
        ]

        work_overlap_pct = (
            len(training_work_days) / training_total_days
        ) * 100

        # 2. Training days that fall on true days off
        training_off_days = [
            day for day in training_days
            if day not in work_days
        ]

        off_blocks = build_off_blocks(bid_days, work_days)

        off_day_to_block = {}

        for block_start, block_end in off_blocks:
            for day in date_range_inclusive(block_start, block_end):
                off_day_to_block[day] = (block_start, block_end)

        middle_penalty_values = []

        for day in training_off_days:
            block = off_day_to_block.get(day)

            if block is None:
                middle_penalty_values.append(1.0)
                continue

            block_start, block_end = block
            block_length = (block_end - block_start).days + 1

            if block_length <= 1:
                middle_penalty_values.append(0.0)
                continue

            days_from_left_edge = (day - block_start).days
            days_from_right_edge = (block_end - day).days

            edge_distance = min(days_from_left_edge, days_from_right_edge)

            max_possible_edge_distance = (block_length - 1) / 2

            middle_penalty = edge_distance / max_possible_edge_distance

            middle_penalty_values.append(middle_penalty)

        if middle_penalty_values:
            off_middle_penalty_pct = (
                sum(middle_penalty_values) / len(middle_penalty_values)
            ) * 100
        else:
            off_middle_penalty_pct = 0.0

        off_edge_score = 100 - off_middle_penalty_pct

        training_fit_score = (
            trip_weight * work_overlap_pct
            + off_edge_weight * off_edge_score
        )

        line["training_fit_score"] = round(training_fit_score, 1)

add_training_fit_score(master_lines,"2023-07-06","2023-07-10",bid_period_info["bid_period_date_range"]["start"],bid_period_info["bid_period_date_range"]["end"])

In [ ]:
from datetime import date, datetime, timedelta


def add_training_fit_score(
    master_lines,
    training_start,
    training_end,
    bid_start,
    bid_end,
    trip_weight=0.80,
    off_edge_weight=0.20,
    category_base_scores=None,
):
    """
    Adds training_fit_score to each line.

    Final score:
        training_fit_score = category_base_score + fit_score

    Category base score:
        Based on the best/highest category touched by the training dates.

    Fit score:
        0 to 100 score that rewards training replacing work/reserve days
        and penalizes training falling in the middle of true days-off blocks.
    """

    if category_base_scores is None:
        category_base_scores = {
            "TRIP": 700,
            "VTO": 200,
            "RB": 600,
            "RA": 500,
            "SB": 400,
            "SA": 300,
            "VOR": 100,
            "UNKNOWN": 0,
        }

    # Codes that should count as "on/work" days for training replacement.
    # VTO is intentionally excluded because it is time off.
    work_codes = {"TRIP", "RB", "RA", "SB", "SA", "VOR"}

    def to_date(value):
        if isinstance(value, datetime):
            return value.date()

        if isinstance(value, date):
            return value

        if isinstance(value, str):
            return date.fromisoformat(value)

        raise TypeError(f"Unsupported date value: {value!r}")

    def date_range_inclusive(start, end):
        current = start
        while current <= end:
            yield current
            current += timedelta(days=1)

    def add_day_category(day_categories, day, category):
        """
        Saves the highest-value category for a given day.

        Example:
            If a day somehow has both TRIP and RB,
            TRIP wins because 700 > 500.
        """
        current_category = day_categories.get(day, "UNKNOWN")

        if category_base_scores.get(category, 0) > category_base_scores.get(current_category, 0):
            day_categories[day] = category

    def get_day_categories_for_line(line):
        """
        Builds a dictionary like:

            {
                date(2023, 7, 6): "TRIP",
                date(2023, 7, 7): "RB",
                date(2023, 7, 8): "VTO",
            }

        True days off will simply not appear and later become UNKNOWN.
        """

        day_categories = {}

        for pp in line.get("PPs", []):
            for assignment in pp.get("assignments", []):

                flights = assignment.get("flights")

                # Case 1: real trip
                if flights:
                    start_dates = [
                        to_date(flight["start_date"])
                        for flight in flights
                        if flight.get("start_date")
                    ]

                    end_dates = [
                        to_date(flight["end_date"])
                        for flight in flights
                        if flight.get("end_date")
                    ]

                    if not start_dates or not end_dates:
                        continue

                    trip_start = min(start_dates)
                    trip_end = max(end_dates)

                    for day in date_range_inclusive(trip_start, trip_end):
                        add_day_category(day_categories, day, "TRIP")

                    continue

                # Case 2: coded assignment such as VTO, RB, RA, SB, SA, VOR
                code = assignment.get("code")
                assignment_date = assignment.get("date")

                if code and assignment_date:
                    code = str(code).strip().upper()

                    if code in category_base_scores:
                        add_day_category(
                            day_categories,
                            to_date(assignment_date),
                            code,
                        )

        return day_categories

    def build_off_blocks(bid_days, work_days):
        off_blocks = []

        current_start = None
        previous_day = None

        for day in sorted(bid_days):
            is_off_day = day not in work_days

            if is_off_day:
                if current_start is None:
                    current_start = day

                previous_day = day

            else:
                if current_start is not None:
                    off_blocks.append((current_start, previous_day))
                    current_start = None
                    previous_day = None

        if current_start is not None:
            off_blocks.append((current_start, previous_day))

        return off_blocks

    training_start = to_date(training_start)
    training_end = to_date(training_end)
    bid_start = to_date(bid_start)
    bid_end = to_date(bid_end)

    if training_end < training_start:
        raise ValueError("training_end must be on or after training_start")

    if bid_end < bid_start:
        raise ValueError("bid_end must be on or after bid_start")

    training_days = list(date_range_inclusive(training_start, training_end))
    bid_days = set(date_range_inclusive(bid_start, bid_end))
    training_total_days = len(training_days)

    for line_num, line in master_lines.items():

        day_categories = get_day_categories_for_line(line)

        # Determine the category for each training day.
        training_day_categories = [
            day_categories.get(day, "UNKNOWN")
            for day in training_days
        ]

        # Pick the best/highest category touched during training.
        best_training_category = max(
            training_day_categories,
            key=lambda category: category_base_scores.get(category, 0),
        )

        category_base_score = category_base_scores.get(best_training_category, 0)

        # Work days are TRIP, RB, RA, SB, SA, VOR.
        # VTO and UNKNOWN are treated as off days.
        work_days = {
            day
            for day, category in day_categories.items()
            if category in work_codes
        }

        # 1. Reward training that overlaps work/reserve days.
        training_work_days = [
            day for day in training_days
            if day in work_days
        ]

        work_overlap_pct = (
            len(training_work_days) / training_total_days
        ) * 100

        # 2. Penalize training that falls in the middle of true days-off blocks.
        training_off_days = [
            day for day in training_days
            if day not in work_days
        ]

        off_blocks = build_off_blocks(bid_days, work_days)

        off_day_to_block = {}

        for block_start, block_end in off_blocks:
            for day in date_range_inclusive(block_start, block_end):
                off_day_to_block[day] = (block_start, block_end)

        middle_penalty_values = []

        for day in training_off_days:
            block = off_day_to_block.get(day)

            if block is None:
                middle_penalty_values.append(1.0)
                continue

            block_start, block_end = block
            block_length = (block_end - block_start).days + 1

            if block_length <= 1:
                middle_penalty_values.append(0.0)
                continue

            days_from_left_edge = (day - block_start).days
            days_from_right_edge = (block_end - day).days

            edge_distance = min(days_from_left_edge, days_from_right_edge)
            max_possible_edge_distance = (block_length - 1) / 2

            middle_penalty = edge_distance / max_possible_edge_distance
            middle_penalty_values.append(middle_penalty)

        if middle_penalty_values:
            off_middle_penalty_pct = (
                sum(middle_penalty_values) / len(middle_penalty_values)
            ) * 100
        else:
            off_middle_penalty_pct = 0.0

        off_edge_score = 100 - off_middle_penalty_pct

        fit_score = (
            trip_weight * work_overlap_pct
            + off_edge_weight * off_edge_score
        )

        line["training_fit_score"] = round(category_base_score + fit_score, 1)


add_training_fit_score(master_lines,"2023-07-06","2023-07-10",bid_period_info["bid_period_date_range"]["start"],bid_period_info["bid_period_date_range"]["end"])

In [ ]:
pprint(master_lines)

In [ ]:
from datetime import date, datetime, timedelta


def count_days_off_around_date(assignments, target_date, before_or_after, bid_start, bid_end):
    """
    Counts consecutive days off immediately before or after a given date.

    Parameters:
        assignments:
            The assignments list from a line or pay period.

        target_date:
            Date to count around. Can be 'YYYY-MM-DD' or date object.

        before_or_after:
            Either 'before' or 'after'.

        bid_start:
            Start date boundary. Can be 'YYYY-MM-DD' or date object.

        bid_end:
            End date boundary. Can be 'YYYY-MM-DD' or date object.

    Returns:
        Integer number of consecutive days off.
    """

    def to_date(value):
        if isinstance(value, date):
            return value
        return datetime.strptime(value, "%Y-%m-%d").date()

    target_date = to_date(target_date)
    bid_start = to_date(bid_start)
    bid_end = to_date(bid_end)

    if before_or_after not in {"before", "after"}:
        raise ValueError("before_or_after must be 'before' or 'after'")

    busy_dates = set()

    for assignment in assignments:

        # Trip assignment
        if "flights" in assignment:
            flight_dates = []

            for flight in assignment["flights"]:
                flight_dates.append(to_date(flight["start_date"]))
                flight_dates.append(to_date(flight["end_date"]))

            trip_start = min(flight_dates)
            trip_end = max(flight_dates)

            current = trip_start
            while current <= trip_end:
                busy_dates.add(current)
                current += timedelta(days=1)

        # Single-day code assignment: RA, RB, SA, SB, VOR, VTO, etc.
        elif "date" in assignment:
            assignment_date = to_date(assignment["date"])
            code = assignment.get("code")

            # RA, RB, SA, SB, VOR, VTO, etc. are not counted as normal days off.
            if code is not None:
                busy_dates.add(assignment_date)

    if before_or_after == "before":
        current = target_date - timedelta(days=1)
        step = -1
    else:
        current = target_date + timedelta(days=1)
        step = 1

    days_off = 0

    while bid_start <= current <= bid_end:
        if current in busy_dates:
            break

        days_off += 1
        current += timedelta(days=step)

    return days_off

In [ ]:
from datetime import date, datetime, timedelta


def add_vacation_days_off_score(
    master_lines,
    vacation_ranges,
    bid_period_info,
    pp_drop_threshold_days=14,
    save_details=True,
):
    """
    Adds a vacation days-off score to each master line.

    The score counts ONLY the extra line-dependent days off directly connected
    to the vacation or dropped pay period.

    It does NOT count:
        - the vacation days themselves
        - the days inside a dropped pay period

    UPS rule:
        If vacation days in a pay period are >= pp_drop_threshold_days,
        that full pay period is treated as protected/dropped.

    Edge cases handled:
        - Vacation causing current PP1 or PP2 to drop.
        - Vacation causing the previous pay period to drop.
        - Vacation causing the next pay period to drop.
        - Multiple separate vacation blocks.
          The function focuses on the largest protected vacation/drop block.
    """
    score_field="extra_vacation_days"

    def to_date(value):
        if isinstance(value, date):
            return value
        return datetime.strptime(value, "%Y-%m-%d").date()

    def make_range(start, end):
        start = to_date(start)
        end = to_date(end)
        return {"start": start, "end": end}

    def count_overlap_days(range_a, range_b):
        start = max(range_a["start"], range_b["start"])
        end = min(range_a["end"], range_b["end"])

        if start > end:
            return 0

        return (end - start).days + 1

    def range_length(date_range):
        return (date_range["end"] - date_range["start"]).days + 1

    def get_all_assignments(line_data):
        assignments = []

        for pp in line_data.get("PPs", []):
            assignments.extend(pp.get("assignments", []))

        return assignments

    def merge_blocks(blocks):
        """
        Merges protected blocks that touch or overlap.

        Example:
            Jun 1-Jun 7 and Jun 8-Jun 14 become Jun 1-Jun 14.
        """

        if not blocks:
            return []

        blocks = sorted(blocks, key=lambda b: b["start"])

        merged = [blocks[0].copy()]

        for block in blocks[1:]:
            last = merged[-1]

            if block["start"] <= last["end"] + timedelta(days=1):
                last["end"] = max(last["end"], block["end"])
                last["reason"] += " + " + block["reason"]
            else:
                merged.append(block.copy())

        return merged

    # Use pay period dates as the real bid boundaries.
    # This avoids accidentally counting the extra day if bid_period end is one day after PP2.
    pp_ranges = {
        pp_name: make_range(pp_info["start"], pp_info["end"])
        for pp_name, pp_info in bid_period_info["pay_period_date_ranges"].items()
    }

    sorted_pps = sorted(pp_ranges.items(), key=lambda item: item[1]["start"])

    bid_start = min(pp["start"] for pp in pp_ranges.values())
    bid_end = max(pp["end"] for pp in pp_ranges.values())

    # Assume pay periods are 28 days unless the actual PP length says otherwise.
    first_pp = sorted_pps[0][1]
    last_pp = sorted_pps[-1][1]
    pp_length = range_length(first_pp)

    previous_pp = {
        "start": first_pp["start"] - timedelta(days=pp_length),
        "end": first_pp["start"] - timedelta(days=1),
    }

    next_pp = {
        "start": last_pp["end"] + timedelta(days=1),
        "end": last_pp["end"] + timedelta(days=pp_length),
    }

    all_pps_to_check = {
        "PREVIOUS_PP": previous_pp,
        **pp_ranges,
        "NEXT_PP": next_pp,
    }

    vacation_blocks = [
        make_range(vac["start"], vac["end"])
        for vac in vacation_ranges
    ]

    protected_blocks = []

    # First add the actual vacation ranges.
    for vac in vacation_blocks:
        protected_blocks.append({
            "start": vac["start"],
            "end": vac["end"],
            "reason": "VACATION",
        })

    # Then apply PP-drop logic.
    for pp_name, pp_range in all_pps_to_check.items():
        vacation_days_in_pp = 0

        for vac in vacation_blocks:
            vacation_days_in_pp += count_overlap_days(vac, pp_range)

        if vacation_days_in_pp >= pp_drop_threshold_days:
            protected_blocks.append({
                "start": pp_range["start"],
                "end": pp_range["end"],
                "reason": f"{pp_name}_DROPPED",
            })

    protected_blocks = merge_blocks(protected_blocks)

    new_vacation_ranges = [
        {
            "start": block["start"].isoformat(),
            "end": block["end"].isoformat(),
        }
        for block in protected_blocks
    ]

    for line_num, line_data in master_lines.items():
        assignments = get_all_assignments(line_data)

        block_scores = []

        for block in protected_blocks:
            days_before = count_days_off_around_date(
                assignments=assignments,
                target_date=block["start"],
                before_or_after="before",
                bid_start=bid_start,
                bid_end=bid_end,
            )

            days_after = count_days_off_around_date(
                assignments=assignments,
                target_date=block["end"],
                before_or_after="after",
                bid_start=bid_start,
                bid_end=bid_end,
            )

            extra_days_off = days_before + days_after
            protected_days = range_length(block)

            block_scores.append({
                "block_start": block["start"],
                "block_end": block["end"],
                "reason": block["reason"],
                "protected_days": protected_days,
                "days_off_before": days_before,
                "days_off_after": days_after,
                "extra_days_off": extra_days_off,
            })

        if block_scores:
            # Important:
            # First prefer the larger vacation/drop block.
            # Then, among similar blocks, prefer more extra days off.
            best_block = max(
                block_scores,
                key=lambda b: (b["protected_days"], b["extra_days_off"])
            )

            line_data[score_field] = best_block["extra_days_off"]

            if save_details:
                line_data[f"{score_field}_details"] = {
                    "selected_block_start": best_block["block_start"].isoformat(),
                    "selected_block_end": best_block["block_end"].isoformat(),
                    "reason": best_block["reason"],
                    "protected_days_not_counted_in_score": best_block["protected_days"],
                    "days_off_before": best_block["days_off_before"],
                    "days_off_after": best_block["days_off_after"],
                    "score": best_block["extra_days_off"],
                }

        else:
            line_data[score_field] = 0

            if save_details:
                line_data[f"{score_field}_details"] = None
    return new_vacation_ranges

In [ ]:
vacation_ranges=[{"start": "2023-05-01", "end": "2023-05-15"},{"start": "2023-07-21", "end": "2023-07-28"}]
nvc = add_vacation_days_off_score(master_lines,vacation_ranges, bid_period_info,pp_drop_threshold_days=14,save_details=False)
nvc

In [ ]:
pprint(master_lines)

In [ ]:
from datetime import date, datetime, timedelta


def add_bid_edge_days_off(
    master_lines,
    bid_period_info,
    edge="both",
    start_field="bid_start_days_off",
    end_field="bid_end_days_off",
):
    """
    Adds days-off counts at the start and/or end of the bid period.

    Uses count_days_off_around_date().

    Parameters:
        master_lines:
            Dictionary of master lines.

        bid_period_info:
            Dictionary containing:
                bid_period_info["bid_period_date_range"]["start"]
                bid_period_info["bid_period_date_range"]["end"]

        edge:
            "start", "end", or "both"

        start_field:
            Field name saved in each line for days off at bid start.

        end_field:
            Field name saved in each line for days off at bid end.

    Returns:
        master_lines, modified in place.
    """

    def to_date(value):
        if isinstance(value, date):
            return value
        return datetime.strptime(value, "%Y-%m-%d").date()

    if edge not in {"start", "end", "both"}:
        raise ValueError("edge must be 'start', 'end', or 'both'")

    bid_start = to_date(bid_period_info["bid_period_date_range"]["start"])
    bid_end = to_date(bid_period_info["bid_period_date_range"]["end"])

    def get_all_assignments(line_data):
        assignments = []

        for pp in line_data.get("PPs", []):
            assignments.extend(pp.get("assignments", []))

        return assignments

    for line_number, line_data in master_lines.items():

        assignments = get_all_assignments(line_data)

        if edge in {"start", "both"}:
            line_data[start_field] = count_days_off_around_date(
                assignments=assignments,
                target_date=bid_start - timedelta(days=1),
                before_or_after="after",
                bid_start=bid_start,
                bid_end=bid_end,
            )

        if edge in {"end", "both"}:
            line_data[end_field] = count_days_off_around_date(
                assignments=assignments,
                target_date=bid_end + timedelta(days=1),
                before_or_after="before",
                bid_start=bid_start,
                bid_end=bid_end,
            )


In [ ]:
add_bid_edge_days_off(master_lines,bid_period_info,edge="both",start_field="bid_start_days_off",end_field="bid_end_days_off",)
pprint(master_lines)
type(master_lines[161]['bid_start_days_off'])